# Software profesional en Acústica 2024-25 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation_Robin](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation_Robin.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install NGSolve. In Google Colab or a fresh environment, run:

In [ ]:
try:
    import ngsolve
except ImportError:
    !pip install ngsolve

# Numerical Solution of the Helmholtz Equation in unbounded domains with Perfectly Matched Layers and a Finite Element Method

This notebook illustrates the numerical solution of the wave equation for an harmonic excitation stated in an unbounded domain using a Finite Element Method. To truncate the unbounded domain, the Perfectly Matched Layer (PML) technique is used. 

## Problem Statement

The homogeneous Helmholtz equation is given as

\begin{equation}
c^2\Delta P(\boldsymbol{x}) + \omega^2 P(\boldsymbol{x}) = 0\qquad \text{for }\quad\boldsymbol{x}=(x,y)\in\Omega_F
\end{equation}

where $\Omega_F$ is the fluid subdomain which where the numerical solution is computed. This subdomain $\Omega_F$ is surrounded by an artificial sponge layer, where the PML goberning equations are stated:

\begin{equation}
c^2\mathrm{div}(C\nabla P(\boldsymbol{x}) + \omega^2 M P(\boldsymbol{x}) = 0\qquad \text{for }\quad\boldsymbol{x}=(x,y)\in\Omega_{PML}
\end{equation}

where
$$
C=
\begin{pmatrix}
\frac{\gamma_y}{\gamma_x} & 0\\
0 & \frac{\gamma_x}{\gamma_y}
\end{pmatrix},
\qquad
M = \gamma_x\gamma_y
$$

with
$$
\gamma_{x}(x)=1+\frac{i}{\omega}\sigma_{x}(s),\qquad \gamma_{y}(y)=1+\frac{i}{\omega}\sigma_{y}(y)
$$

The function $\sigma_x$ and $\sigma_y$ are the so-called PML absorption profiles. These functions can be constant, quadratic or even singular. The only requirements are: $\sigma_x$ and $\sigma_y$ are positive and monotonically increasing.

The set of governing equations must be completed with boundary conditions in the fluid subdomain (in this case, Dirichlet boundary conditions will be used $P=1$ in the interior fluid boundary), and also null Dirichlet boundary conditions will be used in the exterior boundary of the PML subdomain $P=0$. Finally, to couple both fluid and PML subdomains, it is imposed
\begin{align}
u|_{\Omega_{F}}=u|_{\Omega_{PML}}\qquad \text{on }\quad\partial\Omega_{F}\cap \partial\Omega_{PML},\\
\nabla u|_{\Omega_{F}}\cdot \boldsymbol{n}=C\nabla u|_{\Omega_{PML}}\cdot \boldsymbol{n}\qquad \text{on }\quad\partial\Omega_{F}\cap \partial\Omega_{PML},\\
\end{align}

which are the natural coupling boundary conditions to avoid any reflection between both subdomains.

## Variational Formulation

If we extend the definition of the absorption PML profiles in the fluid subdomain as $\sigma_x(x)=0$ and $\sigma_y(y)=0$ for $\boldsymbol{x}=(x,y)\in\Omega_{F}$, then the Helmholtz equation and the PML equation can be written in the same manner in $\Omega=\Omega_F\cup\Omega_{PML}$. Again, as it has been made before, separating the real and the imaginary part of the equations and using a Green's formula, it holds that the pressure field $P=P_{re}+iP_{im}$

\begin{align*}
&c^2\int_\Omega (C_{re}\nabla P_{re}-C_{im}\nabla P_{im})\cdot \nabla Q_{re} \,\mathrm{d}\boldsymbol{x} 
 -\omega^2\int_{\Omega}(M_{re}P_{re}-M_{im}P_{im})Q_{re} \,\mathrm{d}\boldsymbol{x}, \\ 
&c^2\int_\Omega (C_{im}\nabla P_{re}+C_{im}\nabla P_{re})\cdot \nabla Q_{im} \,\mathrm{d}\boldsymbol{x} 
 -\omega^2\int_{\Omega}(M_{im}P_{re}+M_{im}P_{re})Q_{im} \,\mathrm{d}\boldsymbol{x}, 
\end{align*}

for all test functions $Q=Q_{re}+iQ_{im}$ and
where the PML tensor C is given by $C=C_{re}+iC_{im}$ and $M=M_{re}+iM_{im}$.
It is common to express this integral equation in terms of the bilinear $a((P_{re},P_{im}, (Q_{re},Q_{im}))$ and linear $L((Q_{re},Q_{im}))$ forms 

\begin{align*}
a((P_{re},P_{im}, (Q_{re},Q_{im})) = &c^2\int_\Omega (C_{re}\nabla P_{re}-C_{im}\nabla P_{im})\cdot \nabla Q_{re} \,\mathrm{d}\boldsymbol{x}\\
 &+ c^2\int_\Omega (C_{im}\nabla P_{re}+C_{im}\nabla P_{re})\cdot \nabla Q_{im} \,\mathrm{d}\boldsymbol{x} \\
 &-\omega^2\int_{\Omega}(M_{re}P_{re}-M_{im}P_{im})Q_{re} \,\mathrm{d}\boldsymbol{x}\\
 &-\omega^2\int_{\Omega}(M_{im}P_{re}+M_{im}P_{re})Q_{im} \,\mathrm{d}\boldsymbol{x},
\end{align*}
and
\begin{equation*}
L((Q_{re},Q_{im})) = \int_\Omega f_{re}Q_{re} \,\mathrm{d}\boldsymbol{x} + \int_\Omega f_{im}Q_{im} \,\mathrm{d}\boldsymbol{x} ,
\end{equation*}

where

\begin{equation*}
a(P, Q) = L(Q)\qquad \forall Q.
\end{equation*}

In this case, the source term $f=f_{re}+if_{im}=0$.



## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
Its high-level Python interface is used in the following to define the problem and compute the solution.
The implementation is based on the variational formulation derived above.
It is common in the FEM to denote the real and imaginay part of the FEM solution of the problem by $u_{re}$, $u_{im}$ and the corresponding test functions by $v_{re}$ and $v_{im}$.
The definition of the problem in NGSolve is very close to the mathematical formulation of the problem.

#### Import modules and define the physical data and the geometrical setting

In [ ]:
import numpy as np
import scipy.special as spe
import matplotlib.pyplot as plt
from netgen.geom2d import SplineGeometry
from ngsolve import *
from ngsolve.webgui import Draw

# Parameter values
amplitude = 1.0  # amplitude [Pa]
degree = 0       # number of the Fourier mode in the exact solution
omega = 2*np.pi*200.  # angular frequency [rad/s]
vel = 340        # sound speed [m/s]

# Geometrical setting
Radius = 1.0          # radius of a circular obstacle centered at (0,0)
Lx = 2.0; Ly = 2.0   # dimensions of the fluid computational domain
th = 0.75 * vel / (omega / (2.*np.pi))  # PML thickness = 1.5*wavelength

#### Compute mesh

In [ ]:
from netgen.geom2d import SplineGeometry
from netgen.csg import *

# Build geometry using Netgen's SplineGeometry
geo = SplineGeometry()

# Outer boundary of the PML domain (rectangle)
p = [
    geo.AppendPoint(-Lx-th, -Ly-th),
    geo.AppendPoint( Lx+th, -Ly-th),
    geo.AppendPoint( Lx+th,  Ly+th),
    geo.AppendPoint(-Lx-th,  Ly+th),
]
geo.Append(["line", p[0], p[1]], bc="pml_boundary")
geo.Append(["line", p[1], p[2]], bc="pml_boundary")
geo.Append(["line", p[2], p[3]], bc="pml_boundary")
geo.Append(["line", p[3], p[0]], bc="pml_boundary")

# Inner circle (obstacle)
geo.AddCircle(c=(0,0), r=Radius, leftdomain=0, rightdomain=1, bc="circle_boundary")

# Set domain labels and maxh
geo.SetMaterial(1, "fluid_pml")

ngmesh = geo.GenerateMesh(maxh=0.1)
mesh = Mesh(ngmesh)

print("Number of vertices:", mesh.nv)
Draw(mesh)

#### Define the functional spaces and the integral measures

In [ ]:
# Define function space: mixed H1 x H1 for real and imaginary parts
V = H1(mesh, order=1, complex=False, dirichlet="pml_boundary|circle_boundary")
fes = V * V  # product space for (u_re, u_im)

# Define variational unknowns and test functions
(u_re, u_im), (v_re, v_im) = fes.TnT()

#### Define source terms and Dirichlet boundary data

In [ ]:
# Boundary data on the circle: P = cos(n*theta) + i*sin(n*theta)
# g_re = cos(n * atan2(y, x)),  g_im = sin(n * atan2(y, x))
n_mode = degree
g_re = cos(n_mode * atan2(y, x))
g_im = sin(n_mode * atan2(y, x))

# Apply boundary conditions
gfu = GridFunction(fes)
# BCs: on circle_boundary -> (g_re, g_im), on pml_boundary -> (0, 0)
gfu.components[0].Set(g_re, definedon=mesh.Boundaries("circle_boundary"))
gfu.components[1].Set(g_im, definedon=mesh.Boundaries("circle_boundary"))
# pml_boundary is already 0 by default (homogeneous Dirichlet)

#### Define the fluid and the PML coefficients

In [ ]:
# PML absorption profile (quadratic)
sigma0 = 2e3

# sx, sy as NGSolve CoefficientFunctions
sx = IfPos(Abs(x) - Lx,
           sigma0 * (Abs(x) - Lx)**2 / omega,
           0.0)
sy = IfPos(Abs(y) - Ly,
           sigma0 * (Abs(y) - Ly)**2 / omega,
           0.0)

# PML coefficients (real and imaginary parts of gamma_x/gamma_y and gamma_x*gamma_y)
denom_y = sy*sy + 1
denom_x = sx*sx + 1

gammax_div_gammay_re = 1/denom_y + (sx*sy)/denom_y
gammax_div_gammay_im = sx/denom_y - sy/denom_y
gammay_div_gammax_re = 1/denom_x + (sx*sy)/denom_x
gammay_div_gammax_im = sy/denom_x - sx/denom_x
gammax_dot_gammay_re = 1 - sx*sy
gammax_dot_gammay_im = sx + sy

# Physical constants
c2 = vel**2
w2 = omega**2

# PML tensor components (2x2 diagonal matrices as CoefficientFunctions)
C_re = CoefficientFunction((gammay_div_gammax_re, 0, 0, gammax_div_gammay_re), dims=(2,2))
C_im = CoefficientFunction((gammay_div_gammax_im, 0, 0, gammax_div_gammay_im), dims=(2,2))
M_re = gammax_dot_gammay_re
M_im = gammax_dot_gammay_im

#### Define the variational problem and compute the FEM solution

In [ ]:
# Bilinear form
a = BilinearForm(fes)

# Real-part equation
a += c2 * InnerProduct(C_re * grad(u_re) - C_im * grad(u_im), grad(v_re)) * dx
a += -w2 * (M_re*u_re - M_im*u_im) * v_re * dx

# Imaginary-part equation
a += c2 * InnerProduct(C_im * grad(u_re) + C_re * grad(u_im), grad(v_im)) * dx
a += -w2 * (M_im*u_re + M_re*u_im) * v_im * dx

# Linear form (zero source)
f = LinearForm(fes)
f += 0 * v_re * dx
f += 0 * v_im * dx

# Assemble
a.Assemble()
f.Assemble()

# Apply boundary conditions and solve
r = f.vec.CreateVector()
r.data = f.vec - a.mat * gfu.vec
gfu.vec.data += a.mat.Inverse(fes.FreeDofs()) * r

# Extract real and imaginary parts
u_re_sol = gfu.components[0]
u_im_sol = gfu.components[1]

print("Solution computed successfully.")

In [ ]:
# Plot the real part and the magnitude of the pressure field
Draw(u_re_sol, mesh, "Re(P)")
Draw(u_im_sol, mesh, "Im(P)")

# Magnitude
magnitude = sqrt(u_re_sol**2 + u_im_sol**2)
Draw(magnitude, mesh, "|P|")

#### Define the exact solution and compute the $L^2$- relative error in the fluid subdomain

In this case, since the Dirichlet data is an harmonic expression (this is, a complex-exponential $e^{in\theta}$) on the interior circle, the solution is given by
$$
P(r,\theta)=e^{in\theta}H^{(1)}_{n}(kr)
$$
with $k=\omega/c$, $r=\sqrt{x^2+y^2}$, and $\theta=\mathrm{arctan}(y/x)$.

In [ ]:
# Exact solution using a Python callback CoefficientFunction
k = omega / vel
norm_factor = spe.hankel1(degree, k * Radius)

class ExactSolutionCF(CoefficientFunction):
    """NGSolve does not natively support Hankel functions,
    so we use a Python-evaluated GridFunction interpolation."""
    pass

# Interpolate exact solution onto the scalar FE space
Q = H1(mesh, order=3)
uex_re_gf = GridFunction(Q)
uex_im_gf = GridFunction(Q)

# Evaluate exact solution at mesh nodes
coords = mesh.ngmesh.Points()

def exact_re(x_val, y_val):
    r = np.sqrt(x_val**2 + y_val**2)
    if r < Radius * 0.999:
        return 0.0
    theta = np.arctan2(y_val, x_val)
    return np.real(np.exp(1j*degree*theta) * spe.hankel1(degree, k*r) / norm_factor)

def exact_im(x_val, y_val):
    r = np.sqrt(x_val**2 + y_val**2)
    if r < Radius * 0.999:
        return 0.0
    theta = np.arctan2(y_val, x_val)
    return np.imag(np.exp(1j*degree*theta) * spe.hankel1(degree, k*r) / norm_factor)

# Use NGSolve's facility to set a GridFunction from a Python lambda via numpy
uex_re_gf.Set(CoefficientFunction(0))  # initialize
uex_im_gf.Set(CoefficientFunction(0))

# Set via numpy interpolation on the H1 space
from ngsolve import *
uex_re_gf.Set(
    CoefficientFunction(
        IfPos(x**2 + y**2 - Radius**2,  # outside the circle
              # We approximate with a real-part expression via scipy evaluated inline
              # For a full production code, a compiled C++ expression or
              # a table interpolation is preferred.
              # Here we use a workaround: set via numpy on dofs
              CoefficientFunction(0), CoefficientFunction(0))
    )
)

# Proper approach: fill dof values directly
V1 = H1(mesh, order=1)
uex_re_h1 = GridFunction(V1)
uex_im_h1 = GridFunction(V1)

from ngsolve.bla import Vector as NGVector
vals_re = np.array([exact_re(mesh[v].point[0], mesh[v].point[1]) for v in mesh.vertices])
vals_im = np.array([exact_im(mesh[v].point[0], mesh[v].point[1]) for v in mesh.vertices])
uex_re_h1.vec.FV().NumPy()[:] = vals_re
uex_im_h1.vec.FV().NumPy()[:] = vals_im

# Compute relative L2 error in the whole domain
# (fluid subdomain = region where x^2+y^2 > Radius^2)
err_re = u_re_sol - uex_re_h1
err_im = u_im_sol - uex_im_h1

error_num = Integrate(err_re**2 + err_im**2, mesh)
error_den = Integrate(uex_re_h1**2 + uex_im_h1**2, mesh)
error_rel = np.sqrt(error_num / error_den)
print("L2-relative error (%): ", error_rel * 100.)

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).